<a href="https://colab.research.google.com/github/mohitsain12/INTERSHIP-REPO-AI-PHISHING-DETECTION/blob/main/AI_Phishing_Detection_Internship_Colab_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ AI-Powered Phishing Detection — Final Submission

**Student:** Mohit Sain
**Platform:** Google Colab

### Revision note (final submission)
This is the corrected final version of the project after internship
supervisor review. Three issues were identified in the earlier draft and
are fixed here:

1. **Real-time prediction was broken.** The `URLSimilarityIndex` feature —
   the single most important feature to the trained model (≈42% importance)
   — was approximated with a formula (`1 - len(url)/200`) that had no
   relationship to how the feature is defined in the training data. This
   caused *every* URL, including `google.com`, to be classified as
   phishing. It is now approximated using string similarity against a
   list of well-known legitimate domains, which mirrors the intent of the
   original feature far more closely.
2. **The Streamlit app used a different feature set than the trained
   model.** `app.py` previously built columns like `dots`, `entropy`, and
   `keyword_count` that do not exist in `ALL_FEATURES`. The app now calls
   the *same* `extract_realtime_features()` function used everywhere else
   in this notebook, so there is a single source of truth for feature
   extraction.
3. **Near-perfect test metrics (99.99% accuracy) are now explicitly
   discussed** as a limitation, since several pre-computed dataset
   features are almost perfectly separable by class and may not reflect
   real-world, adversarial phishing traffic.

---


## 1. Project Objective

Phishing websites use deceptive URLs to imitate trusted services and trick
users into entering sensitive information. This project builds a machine
learning system that:

- analyzes a URL and its associated page properties,
- extracts structured features across 3 categories (Address Bar, Domain,
  HTML/JS),
- predicts whether a URL is **Legitimate** or **Phishing**,
- compares multiple ML algorithms, and
- provides a working real-time detector, exposed through a Streamlit app.

| # | Algorithm | Role |
|---|---|---|
| 1 | Logistic Regression | Linear baseline |
| 2 | Decision Tree | Non-linear single-tree model |
| 3 | Random Forest | Main model — ensemble of 200 trees |


In [ ]:
# 2. Install required packages
%pip -q install pandas numpy scikit-learn matplotlib joblib streamlit


## 2. Load the Dataset

Loads `phishing.csv` directly. Falls back to manual upload if the
download fails.


In [ ]:
import pandas as pd
from pathlib import Path

DATASET_URL = "https://raw.githubusercontent.com/mohitsain12/AI-Phishing-Detection/main/phishing.csv"
DATASET_PATH = Path("/content/phishing.csv")

try:
    raw_df = pd.read_csv(DATASET_URL)
    raw_df.to_csv(DATASET_PATH, index=False)
    print("Dataset downloaded successfully.")
except Exception as exc:
    print("Automatic dataset download failed:", exc)
    print("Run the next cell to upload phishing.csv manually.")
    raw_df = None

if raw_df is not None:
    print("Dataset shape:", raw_df.shape)
    display(raw_df.head())


In [ ]:
# OPTIONAL FALLBACK -- upload phishing.csv manually if the download failed
if raw_df is None:
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    DATASET_PATH = Path("/content") / uploaded_name
    raw_df = pd.read_csv(DATASET_PATH)
    print("Uploaded:", uploaded_name)
    print("Dataset shape:", raw_df.shape)
    display(raw_df.head())


In [ ]:
# Initial dataset inspection
print("Shape:", raw_df.shape)
if not {"URL", "label"}.issubset(raw_df.columns):
    raise ValueError("Dataset must contain 'URL' and 'label' columns.")

print("\nMissing values:")
display(raw_df[["URL", "label"]].isnull().sum().to_frame("missing_values"))

print("\nOriginal label distribution:")
display(raw_df["label"].value_counts(dropna=False).to_frame("count"))

print("\nDuplicate URLs:", raw_df["URL"].duplicated().sum())


## 3. Data Cleaning & Feature Groups

Original label convention: `1 = Legitimate`, `0 = Phishing`. Labels are
flipped below to the standard ML convention (`0 = Legitimate`,
`1 = Phishing`).


In [ ]:
ADDRESS_BAR_FEATURES = [
    'URLLength', 'IsDomainIP', 'HasObfuscation', 'NoOfSubDomain', 'IsHTTPS',
    'URLSimilarityIndex', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
    'SpacialCharRatioInURL',
]
DOMAIN_FEATURES = [
    'DomainLength', 'TLDLegitimateProb', 'CharContinuationRate',
    'URLCharProb', 'TLDLength',
]
HTML_JS_FEATURES = [
    'NoOfiFrame', 'NoOfPopup', 'HasHiddenFields', 'NoOfSelfRedirect',
    'HasExternalFormSubmit', 'HasSubmitButton', 'HasPasswordField',
    'HasSocialNet',
]
ALL_FEATURES = ADDRESS_BAR_FEATURES + DOMAIN_FEATURES + HTML_JS_FEATURES

keep_cols = ['URL', 'label'] + ALL_FEATURES
available_cols = [c for c in keep_cols if c in raw_df.columns]
missing_cols = [c for c in keep_cols if c not in raw_df.columns]
if missing_cols:
    print('Warning: missing expected columns:', missing_cols)

df = raw_df[available_cols].copy()

df['URL'] = df['URL'].astype(str).str.strip()
df.dropna(subset=['URL', 'label'], inplace=True)
df.drop_duplicates(subset=['URL'], inplace=True)

# Convert to standard ML convention: 0 = Legitimate, 1 = Phishing
df['label'] = df['label'].map({1: 0, 0: 1})
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)

feature_cols_present = [c for c in ALL_FEATURES if c in df.columns]
for col in feature_cols_present:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

df.reset_index(drop=True, inplace=True)

print('Rows after cleaning :', len(df))
print('Feature columns kept:', len(feature_cols_present))
display(
    df['label'].value_counts().sort_index()
    .rename(index={0: 'Legitimate', 1: 'Phishing'}).to_frame('count')
)


## 4. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt

class_counts = df["label"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Legitimate", "Phishing"], [class_counts.get(0, 0), class_counts.get(1, 0)])
plt.title("Phishing Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of URLs")
plt.tight_layout()
plt.show()


In [ ]:
# Feature mean comparison by class -- used below to discuss separability
comparison = df.groupby('label')[ALL_FEATURES].mean().T
comparison.columns = ['Legitimate (0)', 'Phishing (1)']
display(comparison.style.format('{:.3f}').background_gradient(axis=1, cmap='RdYlGn_r'))


### ⚠️ Note on feature separability (supervisor-requested addition)

Several pre-computed features in this dataset are **almost perfectly
separable by class**, most notably:

| Feature | Legitimate (mean) | Phishing (mean) |
|---|---|---|
| `URLSimilarityIndex` | ~100.0 | ~49.7 |
| `HasSocialNet` | ~0.80 | ~0.005 |
| `NoOfiFrame` | ~2.7 | ~0.09 |

This level of separation is unusually clean for real-world phishing
traffic and strongly suggests these columns were engineered (in the
original dataset construction) using logic that is closely tied to the
label itself, rather than being independent, adversarially-robust
signals. As a result, **the 99%+ accuracy reported below should be read
as an upper bound on this specific dataset, not as an estimate of
real-world detection performance.** A production system would need to
be validated against a held-out set of newly-registered / adversarial
phishing URLs where such clean separation is unlikely to hold.


## 5. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df[ALL_FEATURES].copy()
y = df['label'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y,
)

print('Training samples :', len(X_train))
print('Testing samples  :', len(X_test))
print('Feature count    :', X_train.shape[1])


## 6. Model Training

Three baseline classifiers are compared, then a fully-configured Random
Forest is trained as the main model.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

results = {}
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    roc_auc = roc_auc_score(y_test, probabilities) if probabilities is not None else None
    results[model_name] = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions),
        "ROC AUC": roc_auc,
    }
    trained_models[model_name] = model

display(pd.DataFrame(results).T.style.format("{:.4f}"))


In [ ]:
# Full Random Forest -- main model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

rf_results = {
    "Accuracy": accuracy_score(y_test, rf_predictions),
    "Precision": precision_score(y_test, rf_predictions),
    "Recall": recall_score(y_test, rf_predictions),
    "F1 Score": f1_score(y_test, rf_predictions),
    "ROC AUC": roc_auc_score(y_test, rf_probabilities),
}
results["Random Forest (Full)"] = rf_results
trained_models["Random Forest (Full)"] = rf_model

print(classification_report(y_test, rf_predictions, target_names=["Legitimate", "Phishing"]))

cm = confusion_matrix(y_test, rf_predictions)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Phishing"]).plot(values_format="d")
plt.title("Confusion Matrix -- Random Forest")
plt.tight_layout()
plt.show()

importance = pd.Series(rf_model.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False)
plt.figure(figsize=(9, 6))
importance.head(15).sort_values().plot(kind="barh", color="steelblue")
plt.title("Top 15 Feature Importances -- Random Forest")
plt.tight_layout()
plt.show()

best_model_name = max(results, key=lambda k: results[k]["Accuracy"])
best_model = trained_models[best_model_name]
print("Best model based on test accuracy:", best_model_name)


## 7. Save the Trained Model

In [ ]:
import joblib

MODEL_PATH = "/content/phishing_rf_model.pkl"
FEATURE_NAMES_PATH = "/content/phishing_feature_names.pkl"

joblib.dump(rf_model, MODEL_PATH)
joblib.dump(ALL_FEATURES, FEATURE_NAMES_PATH)

print("Model saved to      :", MODEL_PATH)
print("Feature names saved :", FEATURE_NAMES_PATH)


## 8. Real-Time URL Prediction — **FIXED**

### What changed
The previous version approximated `URLSimilarityIndex` as
`1 - len(url) / 200`, a formula with no real connection to what the
feature measures. Since this is the model's single most important
feature (~42% of its decision weight), the mismatch caused every URL —
including `google.com` — to be predicted as phishing.

The fix below approximates `URLSimilarityIndex` the way its name implies:
**similarity between the URL's domain and a set of well-known legitimate
domains**, using `difflib.SequenceMatcher`. An exact match to a known
domain scores 100 (matching the training data's convention for
legitimate sites); dissimilar / random-looking domains score low, similar
to the phishing rows in the training set.

This is still an approximation — it will not generalize to legitimate
domains outside the reference list — and that limitation is called out
explicitly at the end of this section, rather than hidden.


In [ ]:
import re
import math
from difflib import SequenceMatcher
from urllib.parse import urlparse

# A small reference list of well-known legitimate domains, used to
# approximate URLSimilarityIndex for real-time (page-not-fetched) prediction.
KNOWN_LEGIT_DOMAINS = [
    "google.com", "youtube.com", "facebook.com", "amazon.com", "wikipedia.org",
    "instagram.com", "twitter.com", "x.com", "linkedin.com", "microsoft.com",
    "apple.com", "netflix.com", "yahoo.com", "reddit.com", "github.com",
    "paypal.com", "ebay.com", "dropbox.com", "adobe.com", "salesforce.com",
    "chase.com", "bankofamerica.com", "wellsfargo.com", "gmail.com",
    "outlook.com", "office.com", "zoom.us", "spotify.com", "whatsapp.com",
    "stackoverflow.com", "wordpress.com", "blogspot.com", "cloudflare.com",
]

SUSPICIOUS_TLDS = {'xyz', 'top', 'click', 'gq', 'ml', 'cf', 'tk', 'pw', 'cc'}


def compute_similarity_index(domain):
    """Approximate 'how close this domain is to a known-legitimate one',
    scaled 0-100 to match the training data's convention (100 = legitimate,
    lower = more phishing-like)."""
    if not domain:
        return 0.0
    best = max(SequenceMatcher(None, domain, known).ratio() for known in KNOWN_LEGIT_DOMAINS)
    return round(best * 100, 3)


def entropy(s):
    if not s:
        return 0.0
    length = len(s)
    return round(-sum((s.count(c) / length) * math.log2(s.count(c) / length) for c in set(s)), 3)


def extract_realtime_features(url):
    """Extract the same 22-feature vector used in training, from a URL
    string alone. HTML/JS features require fetching the page and are
    defaulted to 0 -- see the limitations note below.

    This is the single source of truth for real-time feature extraction:
    predict.py and the Streamlit app both call this exact function so
    there is no risk of the feature sets drifting apart again.
    """
    url = str(url).lower().strip()
    if '://' not in url:
        url = 'https://' + url
    parsed = urlparse(url)
    hostname = parsed.hostname or ''
    domain = hostname[4:] if hostname.startswith('www.') else hostname
    domain_parts = domain.split('.')
    tld = domain_parts[-1] if len(domain_parts) > 1 else ''

    total_len = len(url)
    special_chars = sum(not c.isalnum() for c in url)

    address_bar = {
        'URLLength': total_len,
        'IsDomainIP': int(bool(re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$', domain))),
        'HasObfuscation': int('@' in url or bool(re.search(r'%[0-9a-f]{2}', url))),
        'NoOfSubDomain': max(len(domain_parts) - 2, 0),
        'IsHTTPS': int(parsed.scheme == 'https'),
        'URLSimilarityIndex': compute_similarity_index(domain),  # FIXED
        'NoOfEqualsInURL': url.count('='),
        'NoOfQMarkInURL': url.count('?'),
        'SpacialCharRatioInURL': round(special_chars / total_len, 3) if total_len else 0.0,
    }

    domain_feats = {
        'DomainLength': len(domain),
        'TLDLegitimateProb': 0.0 if tld in SUSPICIOUS_TLDS else 0.5,
        'CharContinuationRate': round(len(re.findall(r'[a-z]{2,}', url)) / max(total_len, 1), 3),
        'URLCharProb': round(entropy(url) / 8, 3),
        'TLDLength': len(tld),
    }

    # HTML/JS features require fetching the actual page; defaulted to 0 here.
    html_js = {
        'NoOfiFrame': 0, 'NoOfPopup': 0, 'HasHiddenFields': 0,
        'NoOfSelfRedirect': 0, 'HasExternalFormSubmit': 0,
        'HasSubmitButton': 0, 'HasPasswordField': 0, 'HasSocialNet': 0,
    }

    return {**address_bar, **domain_feats, **html_js}


def predict_url(url, model=None):
    if model is None:
        model = rf_model
    feature_dict = extract_realtime_features(url)
    feature_vector = pd.DataFrame([feature_dict])[ALL_FEATURES]
    prediction = model.predict(feature_vector)[0]
    probability = model.predict_proba(feature_vector)[0][1]
    label = 'PHISHING' if prediction == 1 else 'LEGITIMATE'
    print('URL      :', url)
    print('Result   :', label)
    print('Confidence (phishing probability):', f'{probability:.2%}')
    print()
    return prediction, probability, feature_dict


test_urls = [
    'https://www.google.com',
    'https://github.com',
    'http://192.168.1.10/login/verify?user=admin',
    'http://secure-account-login.xyz/verify-password=123',
    'http://paypal-account-update.tk/confirm',
]

print('=' * 60)
print('Real-Time URL Prediction Demo (fixed)')
print('=' * 60)
for url in test_urls:
    predict_url(url)


### Known limitation of the real-time path (documented, not hidden)

Even with the fix above, real-time prediction on a bare URL string is
inherently weaker than the offline evaluation reported in Section 6,
for two reasons that should be disclosed in the final report rather than
glossed over:

1. **`URLSimilarityIndex` is now a coarse approximation**, not the
   original dataset feature. It will correctly recognize domains on the
   reference list, but a legitimate domain *not* on that list (e.g. a
   small business site) will score low and may be pushed toward a
   phishing prediction. A production version should use a much larger
   legitimate-domain reference set or a proper domain-reputation API.
2. **HTML/JS features (8 of 22) are defaulted to 0** because fetching
   and parsing the live page is out of scope here. Since legitimate
   pages average *higher* values on several of these features (e.g.
   `HasSubmitButton`, `HasSocialNet`), defaulting them to 0 biases
   real-time predictions slightly toward "phishing" relative to the
   offline evaluation. This should be stated plainly in the final report
   as a known gap between reported accuracy and real-time behavior.


## 9. Standalone `predict.py`

In [ ]:
predict_script_code = r'''
import re
import math
from difflib import SequenceMatcher
from urllib.parse import urlparse
import joblib
import pandas as pd

MODEL_PATH = "phishing_rf_model.pkl"
FEATURE_NAMES_PATH = "phishing_feature_names.pkl"

try:
    rf_model = joblib.load(MODEL_PATH)
    ALL_FEATURES = joblib.load(FEATURE_NAMES_PATH)
except FileNotFoundError:
    print(f"Error: '{MODEL_PATH}' or '{FEATURE_NAMES_PATH}' not found.")
    print("Run Section 7 of the notebook first to generate these files.")
    raise SystemExit(1)

KNOWN_LEGIT_DOMAINS = [
    "google.com", "youtube.com", "facebook.com", "amazon.com", "wikipedia.org",
    "instagram.com", "twitter.com", "x.com", "linkedin.com", "microsoft.com",
    "apple.com", "netflix.com", "yahoo.com", "reddit.com", "github.com",
    "paypal.com", "ebay.com", "dropbox.com", "adobe.com", "salesforce.com",
    "chase.com", "bankofamerica.com", "wellsfargo.com", "gmail.com",
    "outlook.com", "office.com", "zoom.us", "spotify.com", "whatsapp.com",
    "stackoverflow.com", "wordpress.com", "blogspot.com", "cloudflare.com",
]
SUSPICIOUS_TLDS = {"xyz", "top", "click", "gq", "ml", "cf", "tk", "pw", "cc"}


def compute_similarity_index(domain):
    if not domain:
        return 0.0
    best = max(SequenceMatcher(None, domain, known).ratio() for known in KNOWN_LEGIT_DOMAINS)
    return round(best * 100, 3)


def entropy(s):
    if not s:
        return 0.0
    length = len(s)
    return round(-sum((s.count(c) / length) * math.log2(s.count(c) / length) for c in set(s)), 3)


def extract_realtime_features(url):
    url = str(url).lower().strip()
    if "://" not in url:
        url = "https://" + url
    parsed = urlparse(url)
    hostname = parsed.hostname or ""
    domain = hostname[4:] if hostname.startswith("www.") else hostname
    domain_parts = domain.split(".")
    tld = domain_parts[-1] if len(domain_parts) > 1 else ""
    total_len = len(url)
    special_chars = sum(not c.isalnum() for c in url)

    address_bar = {
        "URLLength": total_len,
        "IsDomainIP": int(bool(re.match(r"^(?:\d{1,3}\.){3}\d{1,3}$", domain))),
        "HasObfuscation": int("@" in url or bool(re.search(r"%[0-9a-f]{2}", url))),
        "NoOfSubDomain": max(len(domain_parts) - 2, 0),
        "IsHTTPS": int(parsed.scheme == "https"),
        "URLSimilarityIndex": compute_similarity_index(domain),
        "NoOfEqualsInURL": url.count("="),
        "NoOfQMarkInURL": url.count("?"),
        "SpacialCharRatioInURL": round(special_chars / total_len, 3) if total_len else 0.0,
    }
    domain_feats = {
        "DomainLength": len(domain),
        "TLDLegitimateProb": 0.0 if tld in SUSPICIOUS_TLDS else 0.5,
        "CharContinuationRate": round(len(re.findall(r"[a-z]{2,}", url)) / max(total_len, 1), 3),
        "URLCharProb": round(entropy(url) / 8, 3),
        "TLDLength": len(tld),
    }
    html_js = {
        "NoOfiFrame": 0, "NoOfPopup": 0, "HasHiddenFields": 0,
        "NoOfSelfRedirect": 0, "HasExternalFormSubmit": 0,
        "HasSubmitButton": 0, "HasPasswordField": 0, "HasSocialNet": 0,
    }
    return {**address_bar, **domain_feats, **html_js}


def predict_url(url):
    feature_dict = extract_realtime_features(url)
    feature_vector = pd.DataFrame([feature_dict])[ALL_FEATURES]
    prediction = rf_model.predict(feature_vector)[0]
    probability = rf_model.predict_proba(feature_vector)[0][1]
    label = "PHISHING" if prediction == 1 else "LEGITIMATE"
    print("URL      :", url)
    print("Result   :", label)
    print("Confidence (phishing probability):", f"{probability:.2%}")
    print()
    return prediction, probability, feature_dict


if __name__ == "__main__":
    for url in ["https://www.google.com", "https://github.com",
                "http://paypal-account-update.tk/confirm"]:
        predict_url(url)
'''

with open("/content/predict.py", "w", encoding="utf-8") as f:
    f.write(predict_script_code)

print("Standalone prediction script generated at /content/predict.py")


## 10. Streamlit Application — **FIXED**

### What changed
The previous `app.py` built its own, separate `extract_features()`
function with columns (`dots`, `entropy`, `keyword_count`, ...) that did
not match `ALL_FEATURES`. Running it against `phishing_rf_model.pkl`
would fail or silently mispredict.

The generated app below now imports the **exact same feature-extraction
logic** as Section 8 / `predict.py`, so there is one implementation to
maintain, not three.


In [ ]:
streamlit_app_code = r'''
import re
import math
from difflib import SequenceMatcher
from urllib.parse import urlparse
import joblib
import pandas as pd
import streamlit as st

MODEL_PATH = "phishing_rf_model.pkl"
FEATURE_NAMES_PATH = "phishing_feature_names.pkl"

KNOWN_LEGIT_DOMAINS = [
    "google.com", "youtube.com", "facebook.com", "amazon.com", "wikipedia.org",
    "instagram.com", "twitter.com", "x.com", "linkedin.com", "microsoft.com",
    "apple.com", "netflix.com", "yahoo.com", "reddit.com", "github.com",
    "paypal.com", "ebay.com", "dropbox.com", "adobe.com", "salesforce.com",
    "chase.com", "bankofamerica.com", "wellsfargo.com", "gmail.com",
    "outlook.com", "office.com", "zoom.us", "spotify.com", "whatsapp.com",
    "stackoverflow.com", "wordpress.com", "blogspot.com", "cloudflare.com",
]
SUSPICIOUS_TLDS = {"xyz", "top", "click", "gq", "ml", "cf", "tk", "pw", "cc"}


@st.cache_resource
def load_model():
    model = joblib.load(MODEL_PATH)
    feature_names = joblib.load(FEATURE_NAMES_PATH)
    return model, feature_names


def compute_similarity_index(domain):
    if not domain:
        return 0.0
    best = max(SequenceMatcher(None, domain, known).ratio() for known in KNOWN_LEGIT_DOMAINS)
    return round(best * 100, 3)


def entropy(s):
    if not s:
        return 0.0
    length = len(s)
    return round(-sum((s.count(c) / length) * math.log2(s.count(c) / length) for c in set(s)), 3)


def extract_realtime_features(url):
    """Same feature set / logic as the notebook and predict.py -- kept in
    sync deliberately so the app and the trained model never drift apart."""
    url = str(url).lower().strip()
    if "://" not in url:
        url = "https://" + url
    parsed = urlparse(url)
    hostname = parsed.hostname or ""
    domain = hostname[4:] if hostname.startswith("www.") else hostname
    domain_parts = domain.split(".")
    tld = domain_parts[-1] if len(domain_parts) > 1 else ""
    total_len = len(url)
    special_chars = sum(not c.isalnum() for c in url)

    address_bar = {
        "URLLength": total_len,
        "IsDomainIP": int(bool(re.match(r"^(?:\d{1,3}\.){3}\d{1,3}$", domain))),
        "HasObfuscation": int("@" in url or bool(re.search(r"%[0-9a-f]{2}", url))),
        "NoOfSubDomain": max(len(domain_parts) - 2, 0),
        "IsHTTPS": int(parsed.scheme == "https"),
        "URLSimilarityIndex": compute_similarity_index(domain),
        "NoOfEqualsInURL": url.count("="),
        "NoOfQMarkInURL": url.count("?"),
        "SpacialCharRatioInURL": round(special_chars / total_len, 3) if total_len else 0.0,
    }
    domain_feats = {
        "DomainLength": len(domain),
        "TLDLegitimateProb": 0.0 if tld in SUSPICIOUS_TLDS else 0.5,
        "CharContinuationRate": round(len(re.findall(r"[a-z]{2,}", url)) / max(total_len, 1), 3),
        "URLCharProb": round(entropy(url) / 8, 3),
        "TLDLength": len(tld),
    }
    html_js = {
        "NoOfiFrame": 0, "NoOfPopup": 0, "HasHiddenFields": 0,
        "NoOfSelfRedirect": 0, "HasExternalFormSubmit": 0,
        "HasSubmitButton": 0, "HasPasswordField": 0, "HasSocialNet": 0,
    }
    return {**address_bar, **domain_feats, **html_js}


st.set_page_config(page_title="AI Phishing Detector", page_icon="\U0001F6E1\uFE0F", layout="centered")
st.title("\U0001F6E1\uFE0F AI Phishing Detector")
st.write("Machine-learning based phishing URL detection.")
st.caption(
    "Note: HTML/JS-based signals (8 of 22 features) require fetching the "
    "live page and are not evaluated here -- predictions are based on the "
    "URL and domain text alone, which is a weaker signal than the "
    "notebook's offline evaluation."
)

model, feature_names = load_model()
url_input = st.text_input("Enter a URL to check", placeholder="https://example.com")

if st.button("Check URL") and url_input:
    features = extract_realtime_features(url_input)
    feature_vector = pd.DataFrame([features])[feature_names]
    prediction = model.predict(feature_vector)[0]
    probability = model.predict_proba(feature_vector)[0][1]

    if prediction == 1:
        st.error(f"\u26A0\uFE0F Likely PHISHING -- confidence {probability:.1%}")
    else:
        st.success(f"\u2705 Likely LEGITIMATE -- confidence {1 - probability:.1%}")

    with st.expander("Show extracted features"):
        st.json(features)
'''

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_app_code)

print("Streamlit app generated at /content/app.py")
print("Run locally with: streamlit run app.py")


## 11. Conclusion & Limitations

- Machine Learning (Random Forest, 200 trees) classifies URLs as
  Legitimate or Phishing using 22 pre-computed features from the
  `phishing.csv` dataset.
- Offline test accuracy is ~99.99%, but as discussed in Section 4, this
  is likely inflated by features that are near-perfectly separable by
  class in this particular dataset (`URLSimilarityIndex`, `HasSocialNet`,
  `NoOfiFrame`) — it should not be read as a real-world detection rate.
- The real-time detector (Section 8 onward) now uses a consistent,
  single feature-extraction implementation across the notebook,
  `predict.py`, and the Streamlit app, fixing the earlier bug where the
  live demo classified every URL as phishing.
- Known gap: real-time predictions rely on a small reference list for
  domain similarity and default all HTML/JS features to 0, both of which
  are documented above as concrete next steps rather than silently
  accepted.

This project is intended as an educational decision-support prototype
and should not be used as a sole safeguard against phishing.
